# Generating the dataset


In [10]:
# This reads the key-value pairs from the .env file and loads them into the current process's environment variables. 
# It returns True if it successfully found and loaded a .env file, or False if it couldn't find one
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# define a Pydantic model for a single exercise

# Each field has a type (str here for all of them) — Pydantic will enforce that.
# When you pass this model to the LLM as a schema, these descriptions get sent along too, essentially as mini-instructions 
# telling the model what should go in each field and what format/example to follow (e.g. "Unique identifier, e.g. 'push-up-001'" 
# nudges the model toward a consistent ID scheme).
# Every field is required (no Optional or default), so the model has to fill in all of them for a valid Exercise.

In [1]:
from pydantic import BaseModel, Field
from typing import List

class Exercise(BaseModel):
    id: str = Field(description="Unique identifier, e.g. 'push-up-001'")
    exercise_name: str = Field(description="Name of the exercise")
    type_of_activity: str = Field(description="Strength, Cardio, Flexibility, etc.")
    type_of_equipment: str = Field(description="Dumbbells, Barbell, None (bodyweight), etc.")
    body_part: str = Field(description="Chest, Back, Legs, etc.")
    type: str = Field(description="Compound, Isolation, etc.")
    muscle_groups_activated: str = Field(description="Comma-separated list, e.g. 'Chest, Triceps, Shoulders'")
    instructions: str = Field(description="Detailed step-by-step instructions")

# This represents many exercises at once — a list of Exercise objects under one exercises field. 
# This is the model you'd actually pass to the LLM call when you want it to generate a whole batch in one go, rather than asking 
# for one exercise at a time.

class ExerciseDataset(BaseModel):
    exercises: List[Exercise]

In [9]:
from openai import OpenAI # imports the OpenAI client class from the openai Python package.
openai_client = OpenAI() # creates an instance of that client, which handles authentication and communication with OpenAI's API.

import pandas as pd # import pandas library — the standard Python tool for working with tabular data

# Set instructions to guide the LLM behavior for the excercise data generation
prompt = """
Generate a dataset of 50 diverse fitness exercises.
Cover different muscle groups, equipment types, and difficulty levels.
Include bodyweight exercises, free weights, and machine exercises.
""".strip()

# For structured output we switch to responses.parse and pass text_format=ExerciseDataset, which tells the API 
# to return our class instead of free text.

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=[{"role": "user", "content": prompt}],
    text_format=ExerciseDataset,
)

dataset = response.output_parsed
df = pd.DataFrame([ex.model_dump() for ex in dataset.exercises])
df.to_csv("data/data.csv", index=False)
print(f"Generated {len(df)} exercises")

Generated 50 exercises


In [7]:
import os
print(os.getcwd())

/Users/vincentaina/fitness-assistant-practice/fitness-assistant


In [8]:
import os
os.makedirs("/Users/vincentaina/fitness-assistant-practice/fitness-assistant/data", exist_ok=True)

In [11]:
df

,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a plank position with hands slightly ...
1,bodyweight-squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with your feet shoulder-width apart and ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves, Core",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Calves, Quadriceps, Shoulders, Core",Stand upright with feet together and arms at y...
5,glute-bridge-006,Glute Bridge,Strength,None (bodyweight),Glutes,Compound,"Glutes, Hamstrings, Core",Lie on your back with knees bent and feet flat...
6,pull-up-007,Pull-Up,Strength,Pull-up Bar,Back,Compound,"Lats, Biceps, Rhomboids, Rear Delts, Core",Grip the pull-up bar slightly wider than shoul...
7,bicep-curl-008,Dumbbell Bicep Curl,Strength,Dumbbells,Arms,Isolation,"Biceps, Forearms",Stand tall with a dumbbell in each hand and pa...
8,shoulder-press-009,Dumbbell Shoulder Press,Strength,Dumbbells,Shoulders,Compound,"Deltoids, Triceps, Upper Chest, Core",Sit or stand holding dumbbells at shoulder hei...
9,bench-press-010,Barbell Bench Press,Strength,Barbell,Chest,Compound,"Chest, Triceps, Shoulders",Lie on a flat bench with your feet planted on ...


In [11]:
df

,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,push-up-001,Push-Up,Strength,None (bodyweight),Chest,Compound,"Chest, Triceps, Shoulders, Core",Start in a plank position with hands slightly ...
1,bodyweight-squat-002,Bodyweight Squat,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Core",Stand with your feet shoulder-width apart and ...
2,plank-003,Plank,Strength,None (bodyweight),Core,Isometric,"Core, Shoulders, Glutes",Place your forearms on the floor with elbows u...
3,lunges-004,Forward Lunge,Strength,None (bodyweight),Legs,Compound,"Quadriceps, Glutes, Hamstrings, Calves, Core",Stand tall with feet hip-width apart. Step for...
4,jumping-jacks-005,Jumping Jacks,Cardio,None (bodyweight),Full Body,Compound,"Calves, Quadriceps, Shoulders, Core",Stand upright with feet together and arms at y...
5,glute-bridge-006,Glute Bridge,Strength,None (bodyweight),Glutes,Compound,"Glutes, Hamstrings, Core",Lie on your back with knees bent and feet flat...
6,pull-up-007,Pull-Up,Strength,Pull-up Bar,Back,Compound,"Lats, Biceps, Rhomboids, Rear Delts, Core",Grip the pull-up bar slightly wider than shoul...
7,bicep-curl-008,Dumbbell Bicep Curl,Strength,Dumbbells,Arms,Isolation,"Biceps, Forearms",Stand tall with a dumbbell in each hand and pa...
8,shoulder-press-009,Dumbbell Shoulder Press,Strength,Dumbbells,Shoulders,Compound,"Deltoids, Triceps, Upper Chest, Core",Sit or stand holding dumbbells at shoulder hei...
9,bench-press-010,Barbell Bench Press,Strength,Barbell,Chest,Compound,"Chest, Triceps, Shoulders",Lie on a flat bench with your feet planted on ...


# The RAG flow

In [13]:
# RAG Architecture - It comes down to three components.
# The pieces are search, the prompt, and the LLM:

# Load the data and create a search index

In [14]:
df = pd.read_csv("data/data.csv")
documents = df.to_dict(orient="records")

In [15]:
# Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. 
# It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. So question, section, and answer 
# are text fields.

# Keyword fields are for exact matching.

from minsearch import Index

index = Index(
    text_fields=[
        "exercise_name",
        "type_of_activity",
        "type_of_equipment",
        "body_part",
        "type",
        "muscle_groups_activated",
        "instructions",
    ],
    keyword_fields=["id"]
)

index.fit(documents)

# implement Search function

In [ ]:
# We do not want to send all documents to LLM cos would be very expensive and not effective 
# We will index the data so only the most relevant docs are retrieved 
# Searching matters because where you have loads of documents. Sending all of them to the LLM would be expensive and slow. 
# The model would get confused with too much data. Search finds the most relevant documents to send instead.

In [17]:
query = "what exercise is similar to push up?"

In [18]:
def search(query):
    boost = {} # no field is treated as more important than another

    results = index.search(
        query=query,
        filter_dict={},  # no filtering. The search runs across everything in the index
        boost_dict=boost,
        num_results=10 # returns the top 10 matches.
    )

    return results

In [20]:
results = search(query)

In [21]:
results

[{'id': 'push-up-001',
  'exercise_name': 'Push-Up',
  'type_of_activity': 'Strength',
  'type_of_equipment': 'None (bodyweight)',
  'body_part': 'Chest',
  'type': 'Compound',
  'muscle_groups_activated': 'Chest, Triceps, Shoulders, Core',
  'instructions': 'Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.'},
 {'id': 'pull-up-007',
  'exercise_name': 'Pull-Up',
  'type_of_activity': 'Strength',
  'type_of_equipment': 'Pull-up Bar',
  'body_part': 'Back',
  'type': 'Compound',
  'muscle_groups_activated': 'Lats, Biceps, Rhomboids, Rear Delts, Core',
  'instructions': 'Grip the pull-up bar slightly wider than shoulder-width with palms facing away. Hang with your arms fully extended and core engaged. Pull your chest toward the bar by driving

# User Prompt Template

In [26]:
USER_PROMPT_TEMPLATE = """

You're a fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

Question:
{query}

Context:
{context}
"""

# Building the context 

In [27]:
# The context is a formatted string with all the search results:

def build_context(results):
    lines = []
    for r in results:
        entry = (
            f"exercise_name: {r['exercise_name']}\n"
            f"type_of_activity: {r['type_of_activity']}\n"
            f"type_of_equipment: {r['type_of_equipment']}\n"
            f"body_part: {r['body_part']}\n"
            f"type: {r['type']}\n"
            f"muscle_groups_activated: {r['muscle_groups_activated']}\n"
            f"instructions: {r['instructions']}"
        )
        lines.append(entry)
    context = "\n\n".join(lines)
    return context

# Building the Prompt

In [32]:
def build_prompt(query, results):
    context = build_context(results)
    prompt = USER_PROMPT_TEMPLATE.format(
        query=query,
        context=context
    )
    return prompt.strip()

In [33]:
prompt = build_prompt(query, results)

In [34]:
print(prompt)

You're a fitness instructor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

Question:
what exercise is similar to push up?

Context:
exercise_name: Push-Up
type_of_activity: Strength
type_of_equipment: None (bodyweight)
body_part: Chest
type: Compound
muscle_groups_activated: Chest, Triceps, Shoulders, Core
instructions: Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form.

exercise_name: Pull-Up
type_of_activity: Strength
type_of_equipment: Pull-up Bar
body_part: Back
type: Compound
muscle_groups_activated: Lats, Biceps, Rhomboids, Rear Delts, Core
instructions: Grip the pull-up bar slightly wider than shoulder-width with palms facing away. Hang with your

# The LLM function

In [35]:
def llm(prompt, model="gpt-5.4-mini"):
    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}]
    )

    return response.output_text

In [36]:
response.output_text # exploring the response

'{"exercises":[{"id":"push-up-001","exercise_name":"Push-Up","type_of_activity":"Strength","type_of_equipment":"None (bodyweight)","body_part":"Chest","type":"Compound","muscle_groups_activated":"Chest, Triceps, Shoulders, Core","instructions":"Start in a plank position with hands slightly wider than shoulder-width apart. Keep your body in a straight line from head to heels. Lower your chest toward the floor by bending your elbows. Pause briefly, then push through your palms to return to the starting position. Repeat with controlled form."},{"id":"bodyweight-squat-002","exercise_name":"Bodyweight Squat","type_of_activity":"Strength","type_of_equipment":"None (bodyweight)","body_part":"Legs","type":"Compound","muscle_groups_activated":"Quadriceps, Glutes, Hamstrings, Core","instructions":"Stand with your feet shoulder-width apart and toes slightly turned out. Brace your core and sit your hips back as if lowering into a chair. Keep your chest up and knees tracking over your toes. Lower u

# Full RAG

In [37]:
# With search, the prompt, and the LLM ready, we wire them together:

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, results)
    answer = llm(prompt, model=model)
    return answer

# Test it

In [38]:
query = "Is the Last Pulldown considered a strength training activity, and if so, why?"
answer = rag(query)
print(answer)

I can’t confirm that from the context provided, because **“Last Pulldown” is not listed in the exercise database**.

If you meant **Pull-Up** or **Assisted Pull-Up Machine**, then **yes**, those are considered **strength training** activities because their `type_of_activity` is **Strength**.
